# Program *Training*: Jaringan Saraf Tiruan dengan *Back Propagation*

Mengenali 3 kelas kendaraan (pesawat, mobil, kapal) dari citra *grayscale* 32 x 32.
Algoritma dan notasi mengikuti Fausett, *Fundamentals of Neural Networks* (1994), Bab 6, Langkah 0 sampai 9.

Hasil akhir: satu berkas `BOBOT.npz` berisi `v`, `v_0`, `w`, `w_0`.
Selama *training* ada tampilan langsung berwarna, lalu animasi yang bisa diputar ulang.

## Sel 1. Persiapan dan pengaturan

Menyambungkan Google Drive dan menetapkan semua parameter di satu tempat. Ubah nilainya di sini bila perlu.

In [ ]:
import os
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display, clear_output, HTML
from google.colab import drive

plt.rcParams['animation.embed_limit'] = 60   # batas ukuran animasi (MB)
drive.mount('/content/drive')

# Lokasi di Google Drive
FOLDER_PROYEK = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI'
FOLDER_LATIH  = FOLDER_PROYEK + '/DATASET/LATIH'
FOLDER_UJI    = FOLDER_PROYEK + '/DATASET/UJI'
BERKAS_BOBOT  = FOLDER_PROYEK + '/BOBOT/BOBOT.npz'

# Ukuran gambar dan arsitektur
LEBAR, TINGGI = 32, 32
n = LEBAR * TINGGI      # jumlah node input
p = 100                 # jumlah node hidden
m = 3                   # jumlah node output (harus tepat 3)

# Parameter pembelajaran
alpha       = 0.1       # laju pembelajaran
EPOCH_MAKS  = 200       # batas epoch (syarat berhenti 1)
AMBANG_GALAT = 0.01     # galat rata-rata per pola (syarat berhenti 2)
BATAS_ACAK  = 0.5       # bobot awal acak di antara -0,5 dan 0,5
BIJI_ACAK   = 42
CETAK_TIAP  = 10        # baris laporan di tabel tiap sekian epoch

# Pengaturan tampilan (tidak mengubah algoritma)
ANIMASI            = True   # False = training tanpa tampilan animasi (lebih cepat)
UPDATE_TIAP        = 5      # perbarui tampilan tiap sekian epoch
JUMLAH_NODE_TAMPIL = 8      # jumlah node hidden yang bobotnya ditampilkan

NAMA_KELAS = ['KELAS1_PESAWAT', 'KELAS2_MOBIL', 'KELAS3_KAPAL']
NAMA_PENDEK = [nama.split('_')[1].capitalize() for nama in NAMA_KELAS]

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Node input  (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p) : {p}')
print(f'Node output (m) : {m}')
print(f'alpha           : {alpha}')
print(f'Epoch maksimum  : {EPOCH_MAKS}')
print(f'Ambang galat    : {AMBANG_GALAT}')
print(f'Animasi         : {"aktif" if ANIMASI else "mati"} (perbarui tiap {UPDATE_TIAP} epoch)')

## Sel 2. *Preprocessing* dan memuat *dataset*

Setiap gambar RGB diubah menjadi *grayscale*, diratakan menjadi vektor berukuran 1024, lalu dinormalisasi ke rentang 0 sampai 1.
Target `t` ditentukan dari nama folder kelas: `1 0 0`, `0 1 0`, `0 0 1`.

In [ ]:
def praproses(gambar):
    """Gambar PIL (RGB) -> vektor x berukuran n, nilai 0 sampai 1."""
    abu = gambar.convert('L')                       # 1. grayscale
    if abu.size != (LEBAR, TINGGI):                 # 2. pengaman ukuran
        abu = abu.resize((LEBAR, TINGGI))
    x = np.asarray(abu, dtype=np.float64).flatten() # 3. ratakan
    return x / 255.0                                # 4. normalisasi


def muat_dataset(folder):
    X, T, label, nama_berkas = [], [], [], []
    for k, nama in enumerate(NAMA_KELAS):
        daftar = sorted(f for f in os.listdir(f'{folder}/{nama}') if f.endswith('.png'))
        for berkas in daftar:
            gambar = Image.open(f'{folder}/{nama}/{berkas}')
            X.append(praproses(gambar))
            target = np.zeros(m)
            target[k] = 1.0                         # 100, 010, atau 001
            T.append(target)
            label.append(k)
            nama_berkas.append(f'{nama}/{berkas}')
    return np.array(X), np.array(T), np.array(label), nama_berkas


X_latih, T_latih, label_latih, _ = muat_dataset(FOLDER_LATIH)
X_uji,   T_uji,   label_uji,   _ = muat_dataset(FOLDER_UJI)

print('=' * 52)
print('DATASET DIMUAT')
print('=' * 52)
print(f'{"Kelas":<18}{"Latih":>8}{"Uji":>8}   Target')
for k, nama in enumerate(NAMA_KELAS):
    target = ' '.join(str(int(a)) for a in np.eye(m)[k])
    print(f'{nama:<18}{(label_latih == k).sum():>8}{(label_uji == k).sum():>8}   {target}')
print('-' * 52)
print(f'{"Total":<18}{len(X_latih):>8}{len(X_uji):>8}')
print(f'\nBentuk X_latih : {X_latih.shape}  (pola x node input)')
print(f'Rentang nilai  : {X_latih.min():.2f} sampai {X_latih.max():.2f}')

## Sel 3. Bukti konversi *grayscale*

Baris atas gambar asli (RGB), baris bawah hasil *grayscale* yang masuk ke jaringan.

In [ ]:
fig, sumbu = plt.subplots(2, 3, figsize=(7, 5))
for k, nama in enumerate(NAMA_KELAS):
    berkas = sorted(f for f in os.listdir(f'{FOLDER_LATIH}/{nama}') if f.endswith('.png'))[0]
    asli = Image.open(f'{FOLDER_LATIH}/{nama}/{berkas}')
    sumbu[0, k].imshow(asli)
    sumbu[0, k].set_title(nama.split('_')[1] + f'\n(mode {asli.mode})')
    sumbu[1, k].imshow(praproses(asli).reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1)
    sumbu[1, k].set_title('grayscale')
    sumbu[0, k].axis('off'); sumbu[1, k].axis('off')
plt.tight_layout()
plt.show()

## Sel 4. Fungsi aktivasi dan inisialisasi bobot (Langkah 0)

Sigmoid biner `f(x) = 1 / (1 + e^-x)` dan turunannya `f'(x) = f(x)(1 - f(x))`.
Semua bobot dan bias diisi bilangan acak kecil.

In [ ]:
def f(x):
    return 1.0 / (1.0 + np.exp(-x))

def f_aksen(x):
    return f(x) * (1.0 - f(x))


# Langkah 0: inisialisasi bobot
rng = np.random.default_rng(BIJI_ACAK)
v   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (n, p))   # input  -> hidden
v_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, p)        # bias hidden
w   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (p, m))   # hidden -> output
w_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, m)        # bias output

print('=' * 52)
print('BOBOT AWAL (Langkah 0)')
print('=' * 52)
print(f'{"Variabel":<10}{"Ukuran":<14}{"Jumlah nilai":>12}')
for nama, larik in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)]:
    print(f'{nama:<10}{str(larik.shape):<14}{larik.size:>12,}')
print(f'\nRentang bobot awal: {min(v.min(), w.min()):.2f} sampai {max(v.max(), w.max()):.2f}')

## Sel 5. *Training* (Langkah 1 sampai 9) dengan tampilan langsung

Algoritma sama seperti sebelumnya: tiap pola dikerjakan dengan Langkah 3 sampai 8, bobot diperbarui per pola.
Tampilan langsung (diperbarui tiap beberapa *epoch*):
- **Atas:** *progress bar* dan waktu.
- **Tengah:** kurva galat, kurva akurasi, dan batang keluaran `y` untuk 3 gambar contoh. Garis hitam adalah target `t`, jadi batang akan mendekati garis hitam seiring *training*.
- **Bawah:** bobot `v` pada beberapa *node hidden* (merah = positif, biru = negatif).

In [ ]:
WARNA_Y = ['#1f77b4', '#ff7f0e', '#9467bd']     # warna batang Y1, Y2, Y3

# ---------- Bahan tampilan (tidak mempengaruhi algoritma) ----------
contoh_x = np.array([X_latih[np.where(label_latih == k)[0][0]] for k in range(m)])
contoh_t = np.eye(m)
node_tampil = np.linspace(0, p - 1, JUMLAH_NODE_TAMPIL).astype(int)
rekaman = []                                   # rekaman keadaan jaringan untuk animasi
riwayat_galat, riwayat_akurasi = [], []


def umpan_maju(X):
    """Hanya untuk melaporkan kemajuan (tidak mengubah bobot)."""
    z = f(v_0 + X @ v)
    return f(w_0 + z @ w)

def akurasi(X, label):
    return np.mean(np.argmax(umpan_maju(X), axis=1) == label) * 100

def simpan_rekaman(epoch):
    rekaman.append({'epoch': epoch, 'waktu': time.time() - waktu_mulai,
                    'y': umpan_maju(contoh_x).copy(), 'bobot': v[:, node_tampil].copy()})

def gambar_dasbor(fig, k, batas_x):
    """Menggambar tampilan pada rekaman ke-k."""
    fig.clf()
    r = rekaman[k]
    e = r['epoch']
    gs = fig.add_gridspec(3, 24, height_ratios=[0.35, 3, 1.7], hspace=0.6, wspace=1.4,
                          left=0.05, right=0.98, top=0.92, bottom=0.07)

    # Progress bar
    a0 = fig.add_subplot(gs[0, :])
    a0.barh([0], [EPOCH_MAKS], color='#e6e6e6')
    a0.barh([0], [e], color='#2ca02c')
    a0.set_xlim(0, EPOCH_MAKS); a0.set_yticks([]); a0.set_xticks([])
    sisa = r['waktu'] / e * (EPOCH_MAKS - e)
    a0.set_title(f'Epoch {e} dari maksimum {EPOCH_MAKS}   |   berlalu {r["waktu"]:.0f} detik'
                 f'   |   sisa paling lama {sisa:.0f} detik', fontsize=10)

    # Kurva galat
    a1 = fig.add_subplot(gs[1, 0:8])
    a1.plot(range(1, e + 1), riwayat_galat[:e], color='#d62728', linewidth=2)
    a1.axhline(AMBANG_GALAT, color='gray', linestyle='--', linewidth=1)
    a1.set_xlim(0, batas_x); a1.set_ylim(0, riwayat_galat[0] * 1.05)
    a1.set_title(f'Galat rata-rata: {riwayat_galat[e - 1]:.4f}', fontsize=10)
    a1.set_xlabel('Epoch'); a1.grid(alpha=0.3)

    # Kurva akurasi
    a2 = fig.add_subplot(gs[1, 8:16])
    a2.plot(range(1, e + 1), riwayat_akurasi[:e], color='#2ca02c', linewidth=2)
    a2.set_xlim(0, batas_x); a2.set_ylim(0, 100)
    a2.set_title(f'Akurasi latih: {riwayat_akurasi[e - 1]:.1f}%', fontsize=10)
    a2.set_xlabel('Epoch'); a2.grid(alpha=0.3)

    # Keluaran y lawan target t untuk 3 gambar contoh
    a3 = fig.add_subplot(gs[1, 16:24])
    for j in range(m):
        geser = (j - 1) * 0.26
        a3.bar(np.arange(m) + geser, r['y'][:, j], width=0.24, color=WARNA_Y[j], label=f'Y{j + 1}')
        a3.hlines(contoh_t[:, j], np.arange(m) + geser - 0.13, np.arange(m) + geser + 0.13,
                  color='black', linewidth=2.5, label='target t' if j == 0 else None)
    a3.set_xticks(range(m)); a3.set_xticklabels(NAMA_PENDEK, fontsize=9)
    a3.set_ylim(0, 1.35); a3.set_yticks([0, 0.5, 1])
    a3.legend(ncol=4, fontsize=7, loc='upper center', frameon=False)
    a3.set_title('Keluaran y (garis hitam = target t)', fontsize=10)

    # Bobot v pada beberapa node hidden
    batas = np.abs(r['bobot']).max()
    for i in range(JUMLAH_NODE_TAMPIL):
        a = fig.add_subplot(gs[2, i * 3:(i + 1) * 3])
        a.imshow(r['bobot'][:, i].reshape(TINGGI, LEBAR), cmap='coolwarm', vmin=-batas, vmax=batas)
        a.axis('off')
        a.set_title(f'hidden {node_tampil[i] + 1}', fontsize=8)
    fig.text(0.5, 0.01, 'Bobot v pada beberapa node hidden (merah = positif, biru = negatif)',
             ha='center', fontsize=9)

def tampilkan_langsung():
    fig = plt.figure(figsize=(13, 7.5))
    gambar_dasbor(fig, len(rekaman) - 1, EPOCH_MAKS)
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


log = []
def catat(epoch, galat, akur):
    if epoch == 1 or epoch % CETAK_TIAP == 0:
        baris = f'{epoch:>6}{galat:>18.5f}{akur:>15.1f}%'
        log.append(baris)
        if not ANIMASI:
            print(baris)

if not ANIMASI:
    print('=' * 52)
    print('TRAINING')
    print('=' * 52)
    print(f'{"Epoch":>6}{"Galat rata-rata":>18}{"Akurasi latih":>16}')
    print('-' * 52)

jumlah_pola = len(X_latih)
waktu_mulai = time.time()

# Langkah 1: ulangi selama syarat berhenti belum terpenuhi
for epoch in range(1, EPOCH_MAKS + 1):
    galat_total = 0.0

    # Langkah 2: untuk setiap pasangan (x, t)
    for s in rng.permutation(jumlah_pola):
        x = X_latih[s]
        t = T_latih[s]

        # Langkah 3 sampai 5: umpan maju
        z_in = v_0 + x @ v
        z    = f(z_in)
        y_in = w_0 + z @ w
        y    = f(y_in)

        # Langkah 6: galat di node output
        delta_k   = (t - y) * f_aksen(y_in)
        delta_w   = alpha * np.outer(z, delta_k)
        delta_w_0 = alpha * delta_k

        # Langkah 7: galat di node hidden (memakai w lama)
        delta_in_j = w @ delta_k
        delta_j    = delta_in_j * f_aksen(z_in)
        delta_v    = alpha * np.outer(x, delta_j)
        delta_v_0  = alpha * delta_j

        # Langkah 8: perbarui bobot dan bias
        w   = w   + delta_w
        w_0 = w_0 + delta_w_0
        v   = v   + delta_v
        v_0 = v_0 + delta_v_0

        galat_total += 0.5 * np.sum((t - y) ** 2)

    galat_rata = galat_total / jumlah_pola
    riwayat_galat.append(galat_rata)
    riwayat_akurasi.append(akurasi(X_latih, label_latih))
    catat(epoch, galat_rata, riwayat_akurasi[-1])

    # Langkah 9: uji syarat berhenti
    berhenti = galat_rata < AMBANG_GALAT
    if ANIMASI and (epoch == 1 or epoch % UPDATE_TIAP == 0 or berhenti or epoch == EPOCH_MAKS):
        simpan_rekaman(epoch)
        tampilkan_langsung()
    if berhenti:
        alasan = f'galat rata-rata di bawah ambang pada epoch {epoch}'
        break
else:
    alasan = f'epoch maksimum ({EPOCH_MAKS}) tercapai'

waktu_total = time.time() - waktu_mulai
print('=' * 52)
print('RINGKASAN TRAINING')
print('=' * 52)
if ANIMASI:
    print(f'{"Epoch":>6}{"Galat rata-rata":>18}{"Akurasi latih":>16}')
    print('-' * 52)
    for baris in log:
        print(baris)
    print('-' * 52)
print(f'Berhenti         : {alasan}')
print(f'Epoch dijalankan : {len(riwayat_galat)}')
print(f'Galat akhir      : {riwayat_galat[-1]:.5f}')
print(f'Akurasi latih    : {riwayat_akurasi[-1]:.1f}%')
print(f'Waktu total      : {waktu_total:.1f} detik')

## Sel 6. Putar ulang animasi *training*

Rekaman proses tadi diputar ulang dengan tombol *play* (atau geser *slider*). Bisa dipakai untuk presentasi tanpa melatih ulang.
Jika `ANIMASI = False`, yang tampil hanya grafik akhir.

In [ ]:
if ANIMASI:
    fig = plt.figure(figsize=(13, 7.5), dpi=60)
    anim = animation.FuncAnimation(fig, lambda k: gambar_dasbor(fig, k, len(riwayat_galat)),
                                   frames=len(rekaman), interval=400, cache_frame_data=False)
    html_animasi = anim.to_jshtml(default_mode='once')
    plt.close(fig)
    display(HTML(html_animasi))
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
    a1.plot(range(1, len(riwayat_galat) + 1), riwayat_galat, color='#d62728')
    a1.set_title('Galat rata-rata per epoch'); a1.set_xlabel('Epoch'); a1.grid(alpha=0.3)
    a2.plot(range(1, len(riwayat_akurasi) + 1), riwayat_akurasi, color='#2ca02c')
    a2.set_title('Akurasi data latih (%)'); a2.set_xlabel('Epoch'); a2.set_ylim(0, 100); a2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Sel 7. Evaluasi pada data latih dan data uji

Data uji tidak ikut dilatih. Selisih besar antara akurasi latih dan uji menandakan *overfitting*.
Pada peta warna, baris adalah kelas sebenarnya dan kolom adalah tebakan jaringan. Hijau = benar, merah = salah.

In [ ]:
tebakan_latih = np.argmax(umpan_maju(X_latih), axis=1)
tebakan_uji   = np.argmax(umpan_maju(X_uji), axis=1)
akurasi_latih = np.mean(tebakan_latih == label_latih) * 100
akurasi_uji   = np.mean(tebakan_uji == label_uji) * 100

print('=' * 52)
print('EVALUASI')
print('=' * 52)
print(f'Akurasi data latih: {akurasi_latih:.1f}%  ({int((tebakan_latih == label_latih).sum())} dari {len(label_latih)} gambar benar)')
print(f'Akurasi data uji  : {akurasi_uji:.1f}%  ({int((tebakan_uji == label_uji).sum())} dari {len(label_uji)} gambar benar)')
print(f'Selisih           : {akurasi_latih - akurasi_uji:.1f} poin')

matriks = np.array([[int(np.sum((label_uji == a) & (tebakan_uji == b))) for b in range(m)] for a in range(m)])

fig, (b1, b2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={'width_ratios': [1, 1.5]})

b1.bar(['Latih', 'Uji'], [akurasi_latih, akurasi_uji], color=['#2ca02c', '#ff7f0e'], width=0.55)
for i, a in enumerate([akurasi_latih, akurasi_uji]):
    b1.text(i, a + 1.5, f'{a:.1f}%', ha='center', fontsize=11)
b1.set_ylim(0, 110); b1.set_title('Akurasi latih lawan uji'); b1.grid(axis='y', alpha=0.3)

warna = np.ones((m, m, 3))
for a in range(m):
    for b in range(m):
        frak = matriks[a, b] / matriks[a].sum()
        dasar = np.array([0.17, 0.63, 0.17]) if a == b else np.array([0.84, 0.15, 0.16])
        warna[a, b] = 1 - frak * (1 - dasar)
        b2.text(b, a, f'{matriks[a, b]}\n{frak * 100:.0f}%', ha='center', va='center', fontsize=11)
b2.imshow(warna)
b2.set_xticks(range(m)); b2.set_xticklabels([f'Tebakan\n{j}' for j in NAMA_PENDEK])
b2.set_yticks(range(m))
b2.set_yticklabels([f'{NAMA_PENDEK[a]}\n({matriks[a, a] / matriks[a].sum() * 100:.0f}% benar)' for a in range(m)])
b2.set_title('Tebakan pada data uji')
plt.tight_layout()
plt.show()

## Sel 8. Pola target dan keluaran jaringan

Tabel pertama menunjukkan pola target `t` tiap kelas. Tabel kedua membandingkan target dengan keluaran jaringan `y` pada gambar uji.
Warna biru makin pekat bila nilai `y` makin besar. Kotak bergaris tebal adalah posisi yang seharusnya bernilai 1.

In [ ]:
def sel_nilai(nilai, target_satu=False):
    alfa = 0.08 + 0.85 * float(nilai)
    gaya = f'background:rgba(31,119,180,{alfa:.2f});'
    if target_satu:
        gaya += 'outline:3px solid #000;outline-offset:-3px;'
    return f'<td style="{gaya}padding:6px 14px;text-align:center;font-family:monospace">{nilai:.3f}</td>'

# Tabel 1: pola target
baris1 = ''
for k in range(m):
    sel_t = ''.join(
        f'<td style="padding:6px 18px;text-align:center;font-weight:bold;'
        f'background:{"#a5d6a7" if a == 1 else "#eeeeee"}">{int(a)}</td>' for a in np.eye(m)[k])
    baris1 += f'<tr><td style="padding:6px 12px">{NAMA_KELAS[k]}</td>{sel_t}</tr>'
html1 = ('<b>Pola target output (t)</b><table style="border-collapse:collapse;margin:6px 0 18px 0">'
         '<tr><th style="padding:6px 12px;text-align:left">Kelas</th><th>Y1</th><th>Y2</th><th>Y3</th></tr>'
         + baris1 + '</table>')

# Tabel 2: keluaran jaringan lawan target (3 gambar uji pertama tiap kelas)
y_uji = umpan_maju(X_uji)
baris2 = ''
for k in range(m):
    for s in np.where(label_uji == k)[0][:3]:
        benar = np.argmax(y_uji[s]) == k
        lencana = ('<span style="background:#2e7d32;color:#fff;padding:2px 10px;border-radius:10px">BENAR</span>' if benar else
                   '<span style="background:#c62828;color:#fff;padding:2px 10px;border-radius:10px">SALAH</span>')
        t_teks = ' '.join(str(int(a)) for a in T_uji[s])
        sel_y = ''.join(sel_nilai(y_uji[s, j], target_satu=(T_uji[s, j] == 1)) for j in range(m))
        baris2 += (f'<tr><td style="padding:6px 12px">{NAMA_PENDEK[k]}</td>'
                   f'<td style="padding:6px 12px;font-family:monospace">{t_teks}</td>{sel_y}'
                   f'<td style="padding:6px 12px">{lencana}</td></tr>')
html2 = ('<b>Keluaran jaringan (y) dibandingkan dengan target (t), data uji</b>'
         '<table style="border-collapse:collapse;margin-top:6px">'
         '<tr><th style="padding:6px 12px;text-align:left">Kelas</th><th>Target t</th>'
         '<th>Y1</th><th>Y2</th><th>Y3</th><th>Hasil</th></tr>' + baris2 + '</table>')
display(HTML(html1 + html2))

## Sel 9. Menyimpan bobot

Bobot `v`, `v_0`, `w`, `w_0` disimpan dalam satu berkas `BOBOT.npz` di folder `BOBOT` pada Google Drive.
Program pengujian akan membaca berkas ini.

In [ ]:
os.makedirs(os.path.dirname(BERKAS_BOBOT), exist_ok=True)
np.savez(BERKAS_BOBOT, v=v, v_0=v_0, w=w, w_0=w_0, ukuran=np.array([LEBAR, TINGGI]))

# Pemeriksaan: baca ulang berkas yang baru disimpan
cek = np.load(BERKAS_BOBOT)
print('=' * 52)
print('BOBOT TERSIMPAN')
print('=' * 52)
print(f'Lokasi : {BERKAS_BOBOT}')
print(f'Ukuran : {os.path.getsize(BERKAS_BOBOT) / 1024:.0f} KB\n')
print(f'{"Isi":<10}{"Ukuran":<14}')
for kunci in ['v', 'v_0', 'w', 'w_0']:
    print(f'{kunci:<10}{str(cek[kunci].shape):<14}')
sama = all(np.array_equal(cek[k], a) for k, a in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)])
print(f'\nIsi berkas sama dengan bobot di memori: {"YA" if sama else "TIDAK"}')